Original Source: https://github.com/openai/openai-cookbook/blob/main/examples/gpt-5/gpt-5_frontend.ipynb

# Frontend with GPT-5.5

GPT-5.5 is a large leap forward in frontend development. We have seen the model be excellent at developing full stack applications in one shot, making complex refactors look easy, and making surgical edits within large codebases. 

In this cookbook we will show some examples and some learnings of developing frontend applications with GPT-5 across multiple axes. 

## Intro
There are some general principles we have seen be effective in developing strong frontend applications. We share some of these learnings in the [prompt guide](https://cookbook.openai.com/examples/gpt-5/gpt-5_prompting_guide). Below are some important pieces to consider when building frontend applications.

Here are libraries and packages we recommend to start with steering the model:
- Frameworks: Next.js (TypeScript), React, HTML
- Styling / UI: Tailwind CSS, shadcn/ui, Radix Themes
- Icons: Material Symbols, Heroicons, Lucide
- Animation: Motion
- Fonts: San Serif, Inter, Geist, Mona Sans, IBM Plex Sans, Manrope

These packages are not an exhaustive list and we have seen many different application styles. 

Below you'll find an easy way to iterate over frontend abstractions on the API. We’re excited to see how users can unlock creativity with GPT-5.


> **Reran against GPT-5.5.** All generations in this notebook now use **`gpt-5.5`**, the current frontier model for coding and frontend work. The original screenshots below were produced with the older GPT-5 model — **re-run each cell with `gpt-5.5` and a valid API key to regenerate fresh outputs**; expect equal-or-better fidelity (tighter theme matching, fewer one-shot fixups).
>
> **TODO — `gpt-5.5-instant` [unverified].** Frontend generation is reasoning-heavy, so it stays on `gpt-5.5`; the instant variant is not used here.

## Interactive Example

Let's dive into an example of creating frontends from scratch. First let's create some help functions to see the generated websites from GPT-5.

In [4]:
import os
import re
import webbrowser
from pathlib import Path

import openai
from openai.types.responses import ResponseInputParam
from openai.types.shared import reasoning_effort

client = openai.OpenAI()


def get_response_output_text(input: str | ResponseInputParam,reasoning_effort="minimal"):
    
    response = client.responses.create(
        model="gpt-5.5",
        input=input,
        reasoning={"effort": reasoning_effort}
    )
    return response.output_text


def extract_html_from_text(text: str):
    """Extract an HTML code block from text; fallback to first code block, else full text."""
    html_block = re.search(r"```html\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    if html_block:
        result = html_block.group(1)
        return result
    any_block = re.search(r"```\s*(.*?)\s*```", text, re.DOTALL)
    if any_block:
        result = any_block.group(1)
        return result
    return text


def save_html(html: str, filename: str) -> Path:
    """Save HTML to outputs/ directory and return the path."""
    try:
        base_dir = Path(__file__).parent
    except NameError:
        base_dir = Path.cwd()

    folder = "outputs"
    outputs_dir = base_dir / folder
    outputs_dir.mkdir(parents=True, exist_ok=True)

    output_path = outputs_dir / filename
    output_path.write_text(html, encoding="utf-8")
    return output_path

def open_in_browser(path: Path) -> None:
    """Open a file in the default browser (macOS compatible)."""
    try:
        webbrowser.open(path.as_uri())
    except Exception:
        os.system(f'open "{path}"')



Now, let's combine the above into one helper function.

In [11]:
def make_website_and_open_in_browser(*, website_input: str | ResponseInputParam, filename: str = "website.html", reasoning_effort: str = "high"):
    response_text = get_response_output_text(website_input, reasoning_effort=reasoning_effort)
    html = extract_html_from_text(response_text)
    output_path = save_html(html, filename)
    open_in_browser(output_path)

We'll start with a simple example: one-shot building a retro gaming store with the right theme

In [ ]:
# Suggestions from students
student_suggestions = [
    "A webapp that syndicates/aggregates a list of RSS feeds and presents the articles based on data (use Hacker News as source for the prototype example).",
    "App to track what items are in storage containers that are placed in storage (picture, barcode, how long in storage); allows searching by picture, use of item, item name.",
    "Maybe have a mini model do a summary for each article in the RSS feeds.",
    "On the storage app: allows you to ask 'I need USD 500, what can I sell and where are the items?'"
]

In [13]:
make_website_and_open_in_browser(
    website_input="A webapp that syndicates/aggregates a list of RSS feeds and presents the articles based on data (use Hacker News as source for the prototype example).",
    filename="MC_rss_feed.html",
)

Not bad for a one line, one shot prompt!

<img src="../../images/retro_dark.png" style="width:60vw; display:block; margin:auto;">


Now let's steer it to be lighter, and a bit softer

As you can see GPT-5 is incredibly steerable - with just a one line you can change entire applications effortlessly

<img src="../../images/retro_light.png" style="width:60vw; display:block; margin:auto;">

But what if you have existing website designs that you want to make additions to? For example, we already have this dashboard.

<img src="../../images/input_image.png" style="width:60vw; display:block; margin:auto;">

Since GPT-5 is natively multimodal and accepts both image and text input, when you are generating frontend applications we can take advantage of image input to improve model performance. 

In [10]:
import base64
from openai.types.responses import ResponseInputImageParam

# Function to encode the image
def encode_image(image_path: str):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

image_path="../../images/input_image.png"
encoded_image = encode_image(image_path)
input_image: ResponseInputImageParam = {"type": "input_image", "image_url": f"data:image/png;base64,{encoded_image}", "detail": "auto"}
input: ResponseInputParam = [
    {
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Can you make a login page for this website that maintains the same theme"},
            input_image,
        ],
    }
]

make_website_and_open_in_browser(
    website_input=input, 
    filename="login_page.html"
)

get_response: finished
extract_html_from_text: finished (raw text)
save_html: finished -> outputs/login_page.html
open_in_browser: finished


As you can see, GPT-5 does an incredible job of matching the existing style & vibe of the app.

<img src="../../images/login_page.png" style="width:60vw; display:block; margin:auto;">

So far this has been pretty static - let's try a more interactive task

In [11]:
make_website_and_open_in_browser(
    website_input="Make me a snake game. It should be futuristic, neon, cyberpunk style. Make sure the typography is suitably cool.", 
    filename="snake_game.html"
)

get_response: finished
extract_html_from_text: finished (raw text)
save_html: finished -> outputs/snake_game.html
open_in_browser: finished


We've got a theme consistent snake game: matching colours, typography, and even sound

<img src="../../images/snake_game.png" style="width:60vw; display:block; margin:auto;">

We hope this has given some ideas of how powerful GPT-5 is at frontend. From a single underspecified prompt and API call, we get production grade outputs. 

Now it's your turn - we can't wait to see what you'll build

In [12]:
your_prompt = "[edit this! what website would you like to build?]"

make_website_and_open_in_browser(
    website_input=your_prompt, 
    filename="your_website.html"
)

get_response: finished
extract_html_from_text: finished (raw text)
save_html: finished -> outputs/your_website.html
open_in_browser: finished


## New Demo 1 — Figma-Export Prompt

Turn an exported Figma design (frames, layers, tokens) into a working component. Paste the Figma export (JSON/Dev-Mode spec or a screenshot) and let GPT-5.5 produce semantic, accessible HTML/CSS that honors the design tokens.

In [ ]:
# Demo 1: Figma export -> production HTML/CSS
figma_export_prompt = """
You are converting a Figma design export into a single self-contained HTML file.

DESIGN TOKENS (from Figma):
- color/primary: #4F46E5
- color/surface: #0B1020
- radius/md: 12px
- spacing/base: 8px
- font/heading: 'Inter', sans-serif

FRAME: "Pricing Card"
- Title, monthly price, 4 feature bullets, primary CTA button.

Rules:
- Honor the tokens EXACTLY (colors, radius, spacing scale).
- Semantic, accessible markup (aria where appropriate). No external CSS frameworks.
- Output ONE ```html code block, nothing else.
"""

make_website_and_open_in_browser(
    website_input=figma_export_prompt,
    filename="figma_pricing_card.html",
    reasoning_effort="high",
)

## New Demo 2 — Data-Science Dashboard Generation

Generate an interactive analytics dashboard from a data description. GPT-5.5 produces the chart scaffolding (e.g. Chart.js / D3 via CDN) plus mock data so the page renders standalone.

In [ ]:
# Demo 2: data-science dashboard from a spec
dashboard_prompt = """
Build a single-file analytics dashboard (HTML + inline JS, Chart.js via CDN).

Data context: a SaaS product's last 12 months of metrics:
- MRR (line chart), churn % (line), signups by plan (stacked bar), NPS (gauge/number).

Requirements:
- Generate realistic MOCK data inline so the page renders with no backend.
- Responsive grid: 2 columns on desktop, 1 on mobile.
- Dark theme, accessible color contrast, a title and last-updated timestamp.
- Output ONE ```html code block only.
"""

make_website_and_open_in_browser(
    website_input=dashboard_prompt,
    filename="ds_dashboard.html",
    reasoning_effort="high",
)

## New Demo 3 — React Component Integration

Instead of a standalone HTML page, ask GPT-5.5 for a drop-in **React** component (TypeScript + props + minimal styles) you can paste into an existing app. Note we read `output_text` directly here rather than rendering HTML in a browser.

In [ ]:
# Demo 3: a drop-in React + TypeScript component
react_prompt = """
Write a single React + TypeScript component named <DataTable/> that:
- Accepts props: columns: {key: string; label: string}[]; rows: Record<string, unknown>[].
- Supports client-side sort on column header click.
- Is fully typed (no `any`), uses only React (no external UI libs).
- Includes a short usage example as a comment.

Implement EXACTLY this — no extra features, no styling framework.
Output ONE ```tsx code block only.
"""

react_component = get_response_output_text(react_prompt, reasoning_effort="high")
print(react_component)